# Help Desk Ticket SLA and Volume Forecasting

This capstone uses `data/raw/issues.csv` for two integrated workstreams: **(1) retrospective comparison of elapsed resolution time with project-configured SLA references by recorded priority; and (2) separate 7- and 30-day daily Ticket-volume forecasting experiments for capacity-planning research.** The 30-day forecast is a daily path, not one monthly total. The primary analysis includes `issue_type == Ticket` and creation dates on/after 2016-01-01 UTC, matching the period described in the accompanying source documentation.

**Important source caveat:** the CSV contains creation dates before 2016 despite the documentation's stated range. Those records are profiled but excluded from the primary scope. Dates without scoped Ticket rows are treated as zero arrivals in the forecasting series under an explicit completeness assumption that has not been independently verified. The source ends in March 2023 and cannot establish current operational performance.

The configured SLA references are project assumptions: Blocker/Highest → Critical (4h), High (8h), Medium (24h), Low/Lowest → Low (24h); `unknown` is excluded. The measured duration is calendar wall-clock time between creation and recorded resolution—not a verified business-hours SLA clock. Results are retrospective, not contractual compliance or a live warning model.

**Shared retrospective result across this notebook sequence:** among 16,735 eligible completed Tickets, 3,333 (19.9%) met the configured reference. By mapped priority: Critical 524/2,124 (24.7%); High 576/3,146 (18.3%); Medium 2,161/11,161 (19.4%); Low 72/304 (23.7%). Eligibility requires exact Ticket type, documented 2016+ scope, `Done`, known mapped priority, and valid nonnegative elapsed duration. Notebook 02 shows the detailed comparison; these same counts and caveats are the project-wide retrospective baseline.


# 01 | Data understanding and retrospective SLA assessment

Profile source completeness, document the date-scope discrepancy, define the ticket cohort and priority mapping, and establish the retrospective reference-attainment denominator. Daily volume aggregation is also created here; the model evaluation is in Notebook 03.

In [2]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from IPython.display import display
from src.issues_capstone import run_data_understanding
result = run_data_understanding()
print(result)


{'source_rows': 66691, 'ticket_rows': 27348, 'sla_assessed_rows': 16735, 'sla_met': 3333, 'daily_observations': 2628, 'reports': WindowsPath('C:/Users/acbitor/PycharmProjects/PythonProject for AIM/output/issue_helpdesk')}


In [3]:
from pathlib import Path
import sys
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from src.issues_capstone import PROJECT_REPORTS, REPORTS
profile = pd.read_csv(PROJECT_REPORTS / "raw_column_profile.csv")
quality = pd.read_csv(PROJECT_REPORTS / "source_quality_summary.csv")
sla = pd.read_csv(PROJECT_REPORTS / "sla_attainment_by_priority.csv")
coverage = pd.read_csv(PROJECT_REPORTS / "daily_coverage_audit.csv")
display(quality)
display(coverage)
display(sla)
display(profile)


,quality_check,value
0,source_rows,66691
1,source_columns,58
2,unique_issue_ids,66691
3,exact_duplicate_rows,0
4,duplicate_project_issue_keys,61
5,documented_scope_start,2016-01-01
6,records_before_documented_scope,23717
7,created_after_resolution_rows,0
8,resolution_disposition_date_mismatch_rows,0
9,resolution_after_last_change_rows,9


,calendar_days,days_with_ticket_arrivals,zero_ticket_days,zero_ticket_day_pct,coverage_status,date_start_utc,date_end_utc
0,2628,2284,344,13.089802,assumed complete from dataset scope; not indep...,2016-01-03,2023-03-14


,priority,assessed_tickets,reference_met,median_elapsed_hours,p90_elapsed_hours,p95_elapsed_hours,reference_hours,reference_met_pct,reference_breach
0,Critical,2124,524,73.493472,958.538556,1465.956750,4.0,24.670433,1600
1,High,3146,576,112.619444,1172.855972,2181.999375,8.0,18.308964,2570
2,Low,304,72,170.996944,2116.562694,4786.611833,24.0,23.684211,232
3,Medium,11161,2161,171.545764,1357.650833,2307.073889,24.0,19.362064,9000
4,Overall,16735,3333,147.259722,1286.016915,2206.733465,NaN,19.916343,13402


,column,dtype,rows_processed,present_rows,missing_rows,missing_pct,unique_values
0,id,float64,66691,66691,0,0.000000,66691
1,started,object,66691,66691,0,0.000000,66281
2,ended,object,66691,66691,0,0.000000,65786
3,issue_num,float64,66691,66691,0,0.000000,3542
4,issue_proj,object,66691,66691,0,0.000000,358
5,issue_reporter,object,66691,66691,0,0.000000,1114
6,issue_assignee,object,66691,35644,31047,46.553508,381
7,issue_contr_count,float64,66691,66691,0,0.000000,9
8,issue_type,object,66691,66691,0,0.000000,15
9,issue_priority,object,66691,66691,0,0.000000,7


## Data dictionary and outlier audit

Per-column data dictionary (type, missingness, uniqueness, meaning/caveat) and an IQR-based outlier audit of elapsed resolution duration for the SLA-assessed cohort.

In [4]:
data_dictionary = pd.read_csv(PROJECT_REPORTS / "data_dictionary.csv")
duration_outliers = pd.read_csv(PROJECT_REPORTS / "resolution_duration_audit.csv")
display(data_dictionary)
display(duration_outliers)

,column,dtype,rows_processed,present_rows,missing_rows,unique_values,meaning_or_caveat
0,id,float64,66691,66691,0,66691,Unique source issue identifier.
1,started,object,66691,66691,0,66281,Workflow-record start timestamp shown in the s...
2,ended,object,66691,66691,0,65786,Workflow-record end/last-change timestamp show...
3,issue_num,float64,66691,66691,0,3542,Issue sequence number within its project; not ...
4,issue_proj,object,66691,66691,0,358,Masked project code associated with the issue.
5,issue_reporter,object,66691,66691,0,1114,Masked user identifier of the issue reporter.
6,issue_assignee,object,66691,35644,31047,381,Masked identifier of the last recorded assigne...
7,issue_contr_count,float64,66691,66691,0,9,Number of users who contributed to the issue.
8,issue_type,object,66691,66691,0,15,Issue kind; primary analysis includes exact va...
9,issue_priority,object,66691,66691,0,7,Recorded priority; unknown is excluded from re...


,rows,min_elapsed_hours,median_elapsed_hours,p95_elapsed_hours,max_elapsed_hours,iqr_low_fence,iqr_high_fence,above_iqr_fence,wf_total_time_duration_spearman
0,16735,0.002778,147.259722,2206.733465,25909.80804,-654.323986,1160.847014,1847,0.99517


**Outlier handling decision:** the 1,847 elapsed-duration values above the IQR high fence are retained, not removed or capped. They are genuine long-running tickets, not measurement errors, and the retrospective SLA-reference comparison already evaluates every eligible ticket against its priority reference regardless of magnitude, so dropping them would hide real breaches. Extreme values are kept in every computed metric for transparency; the priority boxplot in Notebook 02 caps its *y*-axis only for display readability, with that cap stated explicitly in its caption, and the full-range distribution is shown separately alongside it.

## Interpretation guardrails

- Report `Done` outcomes with a mapped known priority and nonnegative elapsed duration; unknown priority is not counted as SLA met or breached.
- The policy references and priority mapping are not independently verified. Validate them with a service owner before compliance claims.
- `wf_total_time` is a separate workflow-derived measure and is only compared as a data diagnostic.
- The source's missing calendar dates are not independently proven to be true no-ticket days.

## In summary

This notebook takes the raw help desk export and checks how complete and trustworthy it is before any modeling happens. It confirms the file mostly covers 2016 through 2023 as documented, with a few older rows set aside, counts how many rows and fields are present or missing, and lays out the ticket priority rules used later in the project. It also produces the baseline result that the rest of the notebooks build on: out of 16,735 tickets eligible for comparison, about 1 in 5 (19.9%) were resolved within their assigned priority target. Think of this notebook as the foundation and fact check step before any forecasting or modeling begins.